In [16]:
# Chemin de la couche Silver enregistrée au format Delta
silver_path = "abfss://healthcare@adlgenstorage.dfs.core.windows.net/Silver/healthcare_clean/"

# Lire les données nettoyées depuis Silver
df_silver = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Nombre de lignes Silver :", df_silver.count())

StatementMeta(sparkhealcare, 4, 3, Finished, Available, Finished, False)

Nombre de lignes Silver : 55106


In [17]:
# Afficher quelques lignes pour vérifier que Silver est lisible

display(df_silver.limit(10))

StatementMeta(sparkhealcare, 4, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9548196-a0dd-4d15-9462-3e02a8def1df)

In [18]:
# GOLD - DIMENSION PATIENT
# Créer une dimension contenant les informations descriptives du patient.

from pyspark.sql.functions import monotonically_increasing_id

dim_patient = (
    df_silver
    .select(
        "name",
        "age",
        "gender",
        "blood_type"
    )
    .dropDuplicates()
)

StatementMeta(sparkhealcare, 4, 5, Finished, Available, Finished, False)

In [19]:
# Ajouter une clé technique unique à chaque patient.
# Cette clé sera utilisée pour relier dim_patient à fact_admission.

dim_patient = dim_patient.withColumn(
    "patient_key",
    monotonically_increasing_id()
)

StatementMeta(sparkhealcare, 4, 6, Finished, Available, Finished, False)

In [20]:
# Afficher quelques patients de la dimension.

display(dim_patient.limit(10))

print("Nombre de patients dans dim_patient :", dim_patient.count())

StatementMeta(sparkhealcare, 4, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87051f89-1acc-47a0-a53c-76538b9c1d90)

Nombre de patients dans dim_patient : 54964


In [21]:
# Afficher quelques patients de la dimension.

display(dim_patient.limit(10))

print("Nombre de patients dans dim_patient :", dim_patient.count())

StatementMeta(sparkhealcare, 4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f9f25e9b-42f8-4bfb-9779-f840a860a1ad)

Nombre de patients dans dim_patient : 54964


In [22]:
# GOLD - DIMENSION HOSPITAL
# Garder une seule ligne pour chaque hôpital.

dim_hospital = (
    df_silver
    .select("hospital")
    .dropDuplicates()
)

StatementMeta(sparkhealcare, 4, 9, Finished, Available, Finished, False)

In [23]:
# Ajouter une clé unique à chaque hôpital.

from pyspark.sql.functions import monotonically_increasing_id

dim_hospital = dim_hospital.withColumn(
    "hospital_key",
    monotonically_increasing_id()
)

StatementMeta(sparkhealcare, 4, 10, Finished, Available, Finished, False)

In [24]:
# GOLD - DIMENSION MEDICAL CONDITION
# Garder une seule ligne pour chaque condition médicale.

dim_condition = (
    df_silver
    .select("medical_condition")
    .dropDuplicates()
)

StatementMeta(sparkhealcare, 4, 11, Finished, Available, Finished, False)

In [25]:
# Ajouter une clé technique à chaque condition médicale.

dim_condition = dim_condition.withColumn(
    "condition_key",
    monotonically_increasing_id()
)

StatementMeta(sparkhealcare, 4, 12, Finished, Available, Finished, False)

In [26]:
display(dim_condition)

StatementMeta(sparkhealcare, 4, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9d1b6b4e-ac0a-4403-9eb4-229744f479c6)

In [27]:
# GOLD - DIMENSION INSURANCE
# Garder une seule ligne pour chaque compagnie d'assurance.

dim_insurance = (
    df_silver
    .select("insurance_provider")
    .dropDuplicates()
)

StatementMeta(sparkhealcare, 4, 14, Finished, Available, Finished, False)

In [28]:
# Ajouter une clé technique à chaque assurance.

dim_insurance = dim_insurance.withColumn(
    "insurance_key",
    monotonically_increasing_id()
)

StatementMeta(sparkhealcare, 4, 15, Finished, Available, Finished, False)

In [29]:
# Vérifier la dimension assurance.

display(dim_insurance)

print("Nombre d'assurances :", dim_insurance.count())

StatementMeta(sparkhealcare, 4, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84713240-3707-4336-9292-5a4294629d48)

Nombre d'assurances : 9


In [30]:
# GOLD - FACT ADMISSION
# Partir des données Silver nettoyées.
# Chaque ligne représente une admission.

fact_admission = df_silver

StatementMeta(sparkhealcare, 4, 17, Finished, Available, Finished, False)

In [31]:
# Relier chaque admission à son patient.
# La jointure permet de récupérer patient_key depuis dim_patient.

fact_admission = fact_admission.join(
    dim_patient.select(
        "patient_key",
        "name",
        "age",
        "gender",
        "blood_type"
    ),
    on=["name", "age", "gender", "blood_type"],
    how="left"
)

StatementMeta(sparkhealcare, 4, 18, Finished, Available, Finished, False)

In [32]:
# Relier chaque admission à la dimension Hospital
# afin de récupérer hospital_key.

fact_admission = fact_admission.join(
    dim_hospital.select(
        "hospital_key",
        "hospital"
    ),
    on="hospital",
    how="left"
)

StatementMeta(sparkhealcare, 4, 19, Finished, Available, Finished, False)

In [33]:
# Relier chaque admission à sa condition médicale
# afin de récupérer condition_key.

fact_admission = fact_admission.join(
    dim_condition.select(
        "condition_key",
        "medical_condition"
    ),
    on="medical_condition",
    how="left"
)

StatementMeta(sparkhealcare, 4, 20, Finished, Available, Finished, False)

In [34]:
# Relier chaque admission à son assurance
# afin de récupérer insurance_key.

fact_admission = fact_admission.join(
    dim_insurance.select(
        "insurance_key",
        "insurance_provider"
    ),
    on="insurance_provider",
    how="left"
)

StatementMeta(sparkhealcare, 4, 21, Finished, Available, Finished, False)

In [35]:
# Calculer la durée du séjour hospitalier en jours.
# discharge_date - date_of_admission = length_of_stay

from pyspark.sql.functions import datediff

fact_admission = fact_admission.withColumn(
    "length_of_stay",
    datediff(
        "discharge_date",
        "date_of_admission"
    )
)

StatementMeta(sparkhealcare, 4, 22, Finished, Available, Finished, False)

In [36]:
# Vérifier quelques admissions après les jointures.

fact_admission.select(
    "patient_key",
    "hospital_key",
    "condition_key",
    "insurance_key",
    "date_of_admission",
    "discharge_date",
    "billing_amount",
    "length_of_stay"
).show(10, truncate=False)

StatementMeta(sparkhealcare, 4, 23, Finished, Available, Finished, False)

+-----------+------------+-------------+-------------+-----------------+--------------+------------------+--------------+
|patient_key|hospital_key|condition_key|insurance_key|date_of_admission|discharge_date|billing_amount    |length_of_stay|
+-----------+------------+-------------+-------------+-----------------+--------------+------------------+--------------+
|11655      |11864       |0            |7            |2022-09-10       |2022-09-14    |18198.50094196973 |4             |
|10193      |9838        |5            |7            |2019-11-04       |2019-11-06    |43063.39672225787 |2             |
|7617       |10707       |5            |8            |2021-06-22       |2021-07-15    |41864.64514931974 |23            |
|12296      |3153        |3            |1            |2020-05-03       |2020-05-24    |39749.06635333829 |21            |
|9077       |11893       |0            |7            |2020-03-29       |2020-04-21    |30715.8379728645  |23            |
|12660      |11454      

In [37]:
# Vérifier le nombre de lignes après les jointures.
# Nous voulons nous assurer que les jointures
# n'ont pas multiplié ou supprimé les admissions.

print("Lignes Silver :", df_silver.count())
print("Lignes Fact :", fact_admission.count())

StatementMeta(sparkhealcare, 4, 24, Finished, Available, Finished, False)

Lignes Silver : 55106
Lignes Fact : 55106


In [38]:
# GOLD - FINALISATION DE FACT_ADMISSION
# Garder les clés étrangères, les mesures
# et les informations propres à l'admission.

fact_admission = fact_admission.select(
    "patient_key",
    "hospital_key",
    "condition_key",
    "insurance_key",
    "date_of_admission",
    "discharge_date",
    "billing_amount",
    "room_number",
    "admission_type",
    "medication",
    "test_results",
    "length_of_stay"
)

StatementMeta(sparkhealcare, 4, 25, Finished, Available, Finished, False)

In [39]:
# Ajouter une clé technique unique à chaque admission.
# admission_key identifie chaque ligne de fact_admission.

from pyspark.sql.functions import monotonically_increasing_id

fact_admission = fact_admission.withColumn(
    "admission_key",
    monotonically_increasing_id()
)

StatementMeta(sparkhealcare, 4, 26, Finished, Available, Finished, False)

In [40]:
# Réorganiser les colonnes de la table Fact.

fact_admission = fact_admission.select(
    "admission_key",
    "patient_key",
    "hospital_key",
    "condition_key",
    "insurance_key",
    "date_of_admission",
    "discharge_date",
    "billing_amount",
    "room_number",
    "admission_type",
    "medication",
    "test_results",
    "length_of_stay"
)

StatementMeta(sparkhealcare, 4, 27, Finished, Available, Finished, False)

In [41]:
# Afficher quelques lignes de la table Fact finale.

display(fact_admission.limit(10))

StatementMeta(sparkhealcare, 4, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0042f902-c31b-4c22-abb0-cfd49bde5dae)

In [43]:
# DATA QUALITY GOLD
# Vérifier que chaque admission a bien trouvé
# une ligne correspondante dans chaque dimension.

from pyspark.sql.functions import col

print(
    "patient_key NULL :",
    fact_admission.filter(col("patient_key").isNull()).count()
)

print(
    "hospital_key NULL :",
    fact_admission.filter(col("hospital_key").isNull()).count()
)

print(
    "condition_key NULL :",
    fact_admission.filter(col("condition_key").isNull()).count()
)

print(
    "insurance_key NULL :",
    fact_admission.filter(col("insurance_key").isNull()).count()
)

StatementMeta(sparkhealcare, 4, 30, Finished, Available, Finished, False)

patient_key NULL : 0
hospital_key NULL : 0
condition_key NULL : 0
insurance_key NULL : 0


In [44]:
# GOLD - Chemins de stockage dans ADLS
# Chaque Fact et Dimension sera enregistrée
# dans son propre dossier au format Delta.

gold_base_path = "abfss://healthcare@adlgenstorage.dfs.core.windows.net/Gold/"

dim_patient_path   = gold_base_path + "dim_patient/"
dim_hospital_path  = gold_base_path + "dim_hospital/"
dim_condition_path = gold_base_path + "dim_condition/"
dim_insurance_path = gold_base_path + "dim_insurance/"
fact_admission_path = gold_base_path + "fact_admission/"

StatementMeta(sparkhealcare, 4, 31, Finished, Available, Finished, False)

In [45]:
# Sauvegarder les dimensions Gold au format Delta.
# overwrite remplace l'ancienne version si nous réexécutons le projet.

dim_patient.write \
    .format("delta") \
    .mode("overwrite") \
    .save(dim_patient_path)

dim_hospital.write \
    .format("delta") \
    .mode("overwrite") \
    .save(dim_hospital_path)

dim_condition.write \
    .format("delta") \
    .mode("overwrite") \
    .save(dim_condition_path)

dim_insurance.write \
    .format("delta") \
    .mode("overwrite") \
    .save(dim_insurance_path)

print("✅ Dimensions Gold sauvegardées")

StatementMeta(sparkhealcare, 4, 32, Finished, Available, Finished, False)

✅ Dimensions Gold sauvegardées


In [46]:
# Sauvegarder la table de faits des admissions
# dans la couche Gold au format Delta.

fact_admission.write \
    .format("delta") \
    .mode("overwrite") \
    .save(fact_admission_path)

print("✅ fact_admission sauvegardée")

StatementMeta(sparkhealcare, 4, 33, Finished, Available, Finished, False)

✅ fact_admission sauvegardée


In [47]:
# Relire fact_admission directement depuis Gold
# pour confirmer que l'écriture Delta fonctionne.

fact_check = (
    spark.read
    .format("delta")
    .load(fact_admission_path)
)

print("Nombre de lignes dans Gold fact_admission :", fact_check.count())

display(fact_check.limit(10))

StatementMeta(sparkhealcare, 4, 34, Finished, Available, Finished, False)

Nombre de lignes dans Gold fact_admission : 55106


SynapseWidget(Synapse.DataFrame, c4509835-1396-4f86-8786-0eb08426494f)